In [ ]:
from pathlib import Path

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pass_pclr.datasets import infer_dataset_class_from_path

In [ ]:
output_dir = Path("../outputs")
datasets = {
    "EchoNext": {
        72475: "runs-echonext",
        32768: "runs-echonext-32k",
        16384: "runs-echonext-16k",
        8192: "runs-echonext-8k",
        4096: "runs-echonext-4k",
        2048: "runs-echonext-2k",
        1024: "runs-echonext-1k",
        512: "runs-echonext-512",
        256: "runs-echonext-256",
    },
    "PTB-XL": {
        17418: "runs-ptbxl",
        8722: "runs-ptbxl-8k",
        4356: "runs-ptbxl-4k",
        2175: "runs-ptbxl-2k",
        1091: "runs-ptbxl-1k",
        547: "runs-ptbxl-512",
        273: "runs-ptbxl-256",
    },
    "CinC Georgia": {
        8192: "runs-cinc",
        4096: "runs-cinc-4k",
        2048: "runs-cinc-2k",
        1024: "runs-cinc-1k",
        512: "runs-cinc-512",
        256: "runs-cinc-256",
    },
}

# mapping of experiment name to tuple of:
# - plotting color
# - folder name
experiments = {
    "LabSup Proto Direct":      ("tab:green",  "proto-from-scratch"),
    "ProtoSSL HEEDB (PIP)":     ("tab:purple", "pass-heedb-pip"),
    "ProtoSSL HEEDB (PIT)":     ("tab:pink",   "pass-heedb-pit"),
    "ProtoSSL HEEDB (PIA)":     ("tab:blue",   "pass-heedb-pit-assign"),
    "LabSup Proto HEEDB (PIP)": ("tab:olive",  "prosup-heedb-pip"),
    "LabSup Proto HEEDB (PIT)": ("tab:cyan",   "prosup-heedb-pit"),
    "LabSup Proto HEEDB (PIA)": ("tab:brown",  "prosup-heedb-pit-assign"),
    "LabSup Proto HEEDB (RIA)": ("tab:orange", "prosup-heedb-pip-then-pit-assign"),
}

echonext_baselines = {
    "Columbia MiniModel": ("black", "columbia-minimodel"),
    "Tabular LR":         ("tab:red", "tabular-logreg-unweighted"),
}

def get_palette(exp_names):
    palette = dict()
    exps = experiments | echonext_baselines
    for exp_name in exp_names:
        temp = exp_name
        if temp.endswith(" LR") and temp != "Tabular LR":
            temp = temp[:-3] # strip suffix
        palette[exp_name] = exps[temp][0]
    return palette

In [ ]:
data = []
for ds, sizes in datasets.items():
    is_echonext = ds == "EchoNext"
    for is_lr in [False, True]:
        for size, run_dir in sizes.items():
            _, labels = infer_dataset_class_from_path(run_dir)
            exps = experiments.copy()
            if is_lr: # get logistic regression results
                exps = {
                    f"{exp_name} LR": (exp_color, f"{exp_dir}-logreg")
                    for exp_name, (exp_color, exp_dir) in exps.items()
                }
            if is_echonext and is_lr: # only get baseline results once (they don't change)
                if size == 72475: # only have minimodel results trained over full echonext size
                    exps["Columbia MiniModel"] = echonext_baselines["Columbia MiniModel"]
                exps["Tabular LR"] = echonext_baselines["Tabular LR"]
            for exp_name, (exp_color, exp_dir) in exps.items():
                metrics = pd.read_csv(output_dir / run_dir / exp_dir / "metrics.csv", index_col="Label")
                multilabel = metrics.loc["Multilabel Averaged"]
                datum = {
                    "Dataset": ds,
                    "Model": exp_name,
                    "Train Size": size,
                    "Multilabel (AUROC)": multilabel["AUROC"],
                    "Multilabel (AUPRC)": multilabel["AUPRC"],
                }
                if is_echonext:
                    composite = metrics.loc["SHD"]
                    datum[f"SHD (AUROC)"] = composite["AUROC"]
                    datum[f"SHD (AUPRC)"] = composite["AUPRC"]
                data.append(datum)
results = pd.DataFrame.from_records(data)

In [ ]:
def plot_lift(
    *,  # enforce kwargs
    df: pd.DataFrame, # long format df (each point to plot is a row)
    dataset: str,
    metric: str,
    models: list[str], # must be intentional about which models to plot
    baseline_model: str | None = None, # singular result to optionally plot as dashed line
    ylim: tuple[float, float] | None = None,
    xlim: tuple[float, float] | None = None,
    save_path: str | None = None,
):
    if baseline_model is not None and baseline_model not in models:
        models = [baseline_model] + models
    palette = get_palette(models)
    df = df[df["Dataset"] == dataset]
    fig, ax = plt.subplots(figsize=(6, 6))

    if baseline_model is not None:
        mask = df["Model"] == baseline_model
        assert (
            mask.sum() == 1
        ), f"Should only have 1 entry for baseline model: {baseline_model}"
        baseline_row = df[mask].iloc[0]
        min_size = df["Train Size"].min()
        max_size = df["Train Size"].max()
        if xlim is not None:
            min_size = min(min_size, xlim[0])
            max_size = max(max_size, xlim[1])
        ax.hlines(
            baseline_row[metric],
            min_size,
            max_size,
            colors=palette.pop(baseline_model),
            linestyles=":",
            label=baseline_model,
        )
        df = df[~mask] # subsequent line plots should exclude baseline model

    sns.lineplot(
        df,
        x="Train Size",
        y=metric,
        hue="Model",
        palette=palette,
        hue_order=list(palette.keys()),
        marker="o",
        ax=ax,
    )
    ax.set_xscale("log", base=2)
    if xlim is not None:
        ax.set_xlim(xlim)
    ax.set_title(f"{dataset} {metric}")
    ax.legend(loc="lower right")
    if ylim is not None:
        ax.set_ylim(ylim)
    if save_path is not None:
        fig.tight_layout()
        fig.savefig(save_path)

In [ ]:
ylim = (0.6, 0.9)

plot_lift(
    df=results,
    dataset="EchoNext",
    metric="Multilabel (AUROC)",
    models=[
        "Tabular LR",
        "LabSup Proto Direct LR",
        "LabSup Proto HEEDB (PIA) LR",
        "LabSup Proto HEEDB (RIA) LR",
        "ProtoSSL HEEDB (PIA) LR",
    ],
    baseline_model="Columbia MiniModel",
    ylim=ylim,
    save_path="figs/echonext-multilabel.png",
)

plot_lift(
    df=results,
    dataset="EchoNext",
    metric="SHD (AUROC)",
    models=[
        "Tabular LR",
        "LabSup Proto Direct LR",
        "LabSup Proto HEEDB (PIA) LR",
        "LabSup Proto HEEDB (RIA) LR",
        "ProtoSSL HEEDB (PIA) LR",
    ],
    baseline_model="Columbia MiniModel",
    ylim=ylim,
    save_path="figs/echonext-composite.png",
)

plot_lift(
    df=results,
    dataset="PTB-XL",
    metric="Multilabel (AUROC)",
    models=[
        "Tabular LR",
        "LabSup Proto Direct LR",
        "LabSup Proto HEEDB (PIA) LR",
        "LabSup Proto HEEDB (RIA) LR",
        "ProtoSSL HEEDB (PIA) LR",
    ],
    ylim=ylim,
    save_path="figs/ptbxl-multilabel.png",
)

plot_lift(
    df=results,
    dataset="CinC Georgia",
    metric="Multilabel (AUROC)",
    models=[
        "Tabular LR",
        "LabSup Proto Direct LR",
        "LabSup Proto HEEDB (PIA) LR",
        "LabSup Proto HEEDB (RIA) LR",
        "ProtoSSL HEEDB (PIA) LR",
    ],
    ylim=ylim,
    save_path="figs/cinc-multilabel.png",
)